# Cafe Sales Data Cleaning
**Student Name:** Mohammed Al-Zubiri

## Step 1: Load the Dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('dirty_cafe_sales.csv')
print("Shape:", df.shape)
df.head()

Shape: (10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


Checking data types and missing values.

In [2]:
print(df.dtypes)
print()
print(df.isnull().sum())

Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64


All columns are `object` type because of invalid strings mixed in. Checking for ERROR and UNKNOWN entries.

In [3]:
for col in df.columns:
    for val in ['ERROR', 'UNKNOWN']:
        count = (df[col] == val).sum()
        if count > 0:
            print(f"{col} has {count} '{val}' entries")

Item has 292 'ERROR' entries
Item has 344 'UNKNOWN' entries
Quantity has 170 'ERROR' entries
Quantity has 171 'UNKNOWN' entries
Price Per Unit has 190 'ERROR' entries
Price Per Unit has 164 'UNKNOWN' entries
Total Spent has 164 'ERROR' entries
Total Spent has 165 'UNKNOWN' entries
Payment Method has 306 'ERROR' entries
Payment Method has 293 'UNKNOWN' entries
Location has 358 'ERROR' entries
Location has 338 'UNKNOWN' entries
Transaction Date has 142 'ERROR' entries
Transaction Date has 159 'UNKNOWN' entries


## Step 2: Replace ERROR and UNKNOWN with NaN
These are invalid placeholders, replacing with NaN so pandas can handle them.

In [4]:
df.replace(['ERROR', 'UNKNOWN'], np.nan, inplace=True)

## Step 3: Data Type Correction
After removing invalid strings, converting numeric and date columns to correct types.

In [5]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

print(df.dtypes)

Transaction ID              object
Item                        object
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object


## Step 4: Fix Item Names Using Price Per Unit
Each item has a fixed price. Some prices map to only one item so we can recover missing names:
- Cookie = 1.0
- Tea = 1.5
- Coffee = 2.0
- Salad = 5.0

Prices 3.0 (Cake or Juice) and 4.0 (Sandwich or Smoothie) map to two items each, can't determine those.

In [6]:
price_to_item = {
    1.0: 'Cookie',
    1.5: 'Tea',
    2.0: 'Coffee',
    5.0: 'Salad'
}

missing_item = df['Item'].isna()
for price, item in price_to_item.items():
    mask = missing_item & (df['Price Per Unit'] == price)
    df.loc[mask, 'Item'] = item

## Step 5: Fill Missing Price Per Unit and Total Spent
1. Fill Price Per Unit from item name since each item has a known price.
2. Use `Total Spent = Quantity * Price Per Unit` to fill whichever value is missing when the other two exist.

In [7]:
# fill Price Per Unit from item name
item_to_price = {
    'Coffee': 2.0, 'Cake': 3.0, 'Cookie': 1.0, 'Juice': 3.0,
    'Salad': 5.0, 'Sandwich': 4.0, 'Smoothie': 4.0, 'Tea': 1.5
}
for item, price in item_to_price.items():
    mask = (df['Item'] == item) & df['Price Per Unit'].isna()
    df.loc[mask, 'Price Per Unit'] = price

# calculate missing Total Spent
mask = df['Total Spent'].isna() & df['Quantity'].notna() & df['Price Per Unit'].notna()
df.loc[mask, 'Total Spent'] = df.loc[mask, 'Quantity'] * df.loc[mask, 'Price Per Unit']

# calculate missing Quantity
mask = df['Quantity'].isna() & df['Total Spent'].notna() & df['Price Per Unit'].notna()
df.loc[mask, 'Quantity'] = df.loc[mask, 'Total Spent'] / df.loc[mask, 'Price Per Unit']

# calculate missing Price Per Unit from the other two
mask = df['Price Per Unit'].isna() & df['Total Spent'].notna() & df['Quantity'].notna()
df.loc[mask, 'Price Per Unit'] = df.loc[mask, 'Total Spent'] / df.loc[mask, 'Quantity']

## Step 6: Handle Categorical Columns
Payment Method and Location have too many missing values (2500+ and 3200+) to drop. Filling with "Unknown" instead.

In [8]:
df['Payment Method'] = df['Payment Method'].fillna('Unknown')
df['Location'] = df['Location'].fillna('Unknown')

## Step 7: Drop Rows With Remaining Missing Values
Rows still missing Transaction Date, Item, Quantity, Price Per Unit, or Total Spent can't be recovered. Dropping them.

In [9]:
df.dropna(subset=['Transaction Date', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent'], inplace=True)

df['Quantity'] = df['Quantity'].astype(int)

print("Remaining nulls:")
print(df.isnull().sum())

Remaining nulls:
Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64


## Step 8: Create Season Column
Season derived from transaction month:
- Winter: December, January, February
- Spring: March, April, May
- Summer: June, July, August
- Fall: September, October, November

In [10]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['Season'] = df['Transaction Date'].dt.month.apply(get_season)
df['Season'].value_counts()

Season
Fall      2300
Summer    2277
Spring    2258
Winter    2209
Name: count, dtype: int64

## Step 9: Save Cleaned Data

In [11]:
df.to_csv('cleaned_cafe_sales.csv', index=False)

print("Cleaned shape:", df.shape)
print()
print(df.dtypes)
print()
df.head(10)

Cleaned shape: (9044, 9)

Transaction ID              object
Item                        object
Quantity                     int64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
Season                      object
dtype: object



,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Season
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08,Fall
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16,Spring
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19,Summer
3,TXN_7034554,Salad,2,5.0,10.0,Unknown,Unknown,2023-04-27,Spring
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11,Summer
5,TXN_2602893,Smoothie,5,4.0,20.0,Credit Card,Unknown,2023-03-31,Spring
7,TXN_6699534,Sandwich,4,4.0,16.0,Cash,Unknown,2023-10-28,Fall
9,TXN_2064365,Sandwich,5,4.0,20.0,Unknown,In-store,2023-12-31,Winter
10,TXN_2548360,Salad,5,5.0,25.0,Cash,Takeaway,2023-11-07,Fall
12,TXN_7619095,Sandwich,2,4.0,8.0,Cash,In-store,2023-05-03,Spring
